# 06. Structured Streaming and Checkpointing

## Banking Transaction Lakehouse

**Purpose:** Demonstrate incremental file-based transaction processing using Spark Structured Streaming.

### Scope
- Read incoming Parquet files using `readStream`.
- Write processed records into a Delta destination.
- Configure checkpoint locations.
- Use the `AvailableNow` trigger.
- Process newly arriving files.
- Restart the query using the same checkpoint.
- Verify streaming progress and output records.

**Implementation:** This is a standalone streaming demonstration. It is not yet integrated with the main batch orchestration pipeline.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Row

# Dedicated paths for this notebook
base_path = "/Volumes/dbacademy/default/raw_data/streaming_notebook_demo"

input_path = f"{base_path}/input"
output_path = f"{base_path}/output"
checkpoint_path = f"{base_path}/checkpoint"

print("Structured Streaming notebook initialized.")
print("Input path:", input_path)
print("Output path:", output_path)
print("Checkpoint path:", checkpoint_path)

In [0]:
# Create initial transaction records
initial_data = [
    Row(transaction_id=2001, customer_id=101, amount=1500.0, transaction_type="DEBIT"),
    Row(transaction_id=2002, customer_id=102, amount=2500.0, transaction_type="CREDIT"),
    Row(transaction_id=2003, customer_id=103, amount=3500.0, transaction_type="DEBIT")
]

initial_df = spark.createDataFrame(initial_data)

# Write initial records as Parquet files
(
    initial_df.write
    .format("parquet")
    .mode("overwrite")
    .save(input_path)
)

print("Initial source records created:", initial_df.count())
display(initial_df)

In [0]:
# Define streaming source
stream_df = (
    spark.readStream
    .format("parquet")
    .schema(initial_df.schema)
    .load(input_path)
)

print("Is streaming DataFrame:", stream_df.isStreaming)
print("Streaming source schema:")

stream_df.printSchema()

In [0]:
# Start streaming query and write results to Delta
query = (
    stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start(output_path)
)

# Wait until all available data is processed
query.awaitTermination()

print("Streaming query completed.")

In [0]:
# Read processed streaming output
stream_output_df = spark.read.format("delta").load(output_path)

print("Total processed records:", stream_output_df.count())

display(stream_output_df.orderBy("transaction_id"))

In [0]:
# Simulate newly arriving transaction records
new_data = [
    Row(transaction_id=2004, customer_id=104, amount=4500.0, transaction_type="CREDIT"),
    Row(transaction_id=2005, customer_id=105, amount=5500.0, transaction_type="DEBIT")
]

new_df = spark.createDataFrame(
    new_data,
    initial_df.schema
)

# Append new Parquet files to the source directory
(
    new_df.write
    .format("parquet")
    .mode("append")
    .save(input_path)
)

print("New transactions added:", new_df.count())
display(new_df)

In [0]:
# Restart the streaming query using the existing checkpoint
query = (
    stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start(output_path)
)

query.awaitTermination()

print("Incremental streaming run completed.")

In [0]:
# Read the final Delta output
final_output_df = (
    spark.read
    .format("delta")
    .load(output_path)
)

print("Total processed records:", final_output_df.count())

display(
    final_output_df.orderBy("transaction_id")
)